In [ ]:
import json
import urllib.request
import urllib.parse
import xml.etree.ElementTree as ET
import re
import time
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from google.colab import userdata

# --- 1. CONFIGURATION ---
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    GEMINI_API_KEY = input("Paste your Gemini API Key here: ").strip()

USER_INTERESTS = "Unexplained Mysteries, Historical Curiosities, Scientific Facts, True Crime, Weird News, Lendas Urbanas"

RSS_FEEDS = [
    "https://www.reddit.com/r/Damnthatsinteresting/top/.rss?t=day",
    "https://www.reddit.com/r/todayilearned/top/.rss?t=day",
    "https://www.reddit.com/r/ExplainLikeImFive/top/.rss?t=day",
    "https://www.reddit.com/r/Creepy/top/.rss?t=day",
    "https://www.reddit.com/r/OddlyTerrifying/top/.rss?t=day",
    "https://www.reddit.com/r/BeAmazed/top/.rss?t=day",
    "https://listverse.com/feed/",
    "https://www.boredpanda.com/feed/",
    "https://www.thefactsite.com/feed/",
    "https://www.mentalfloss.com/rss.xml",
    "https://www.smithsonianmag.com/rss/smart-news/",
    "https://www.livescience.com/feeds/all",
    "https://www.damninteresting.com/feed/",
    "https://super.abril.com.br/feed/",
    "https://super.abril.com.br/mundo-estranho/feed/",
    "https://gizmodo.uol.com.br/feed/",
    "https://aventurasnahistoria.com.br/feed/",
    "https://canaltech.com.br/rss/ciencia/",
    "https://www.youtube.com/feeds/videos.xml?channel_id=UC7zbUfFoMAMGHIRUE8gjVnw", # TechTudo
    "http://astronomy-universo.blogspot.com/feeds/posts/default", # Astronomia e Universo
    "http://www.numaniaticos.com/feed/", # Numaniáticos
    "https://api.reddit.com/subreddit/creepy", # r/creepy
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCVVIVc6DfU8uWJ60vk8iKpQ", # Superinteressante
    "https://www.youtube.com/feeds/videos.xml?channel_id=UChz45Ir-nTCxn7qmJ5nXQ6Q", # GizmodoBR
    "http://www.vocesabia.net/feed/", # Curiosidades no Você Sabia
    "http://super.abril.com.br/blogs/oraculo/feed/", # Oráculo – Super
    "https://hypescience.com/feed/", # HypeScience
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCH2VZQBLFTOp6I_qgnBJCuQ", # Canal Nostalgia
    "http://randomc.net/feed/", # Random Curiosity
    "https://www.youtube.com/feeds/videos.xml?channel_id=UClu474HMt895mVxZdlIHXEA", # Nerdologia
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCunqx90PNIv30uOgn0m4CTg", # Acredite ou Não
    "http://scienceblogs.com.br/colecionadores/feed/", # Colecionadores de Ossos
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCNYaxPiba3oxmeL_3jKxnYA", # Insane Curiosity
    "http://rss.megacurioso.com.br/feed", # Novidades do Mega Curioso
    "http://gdata.youtube.com/feeds/base/users/superinteressanteweb/uploads?alt=rss&v=2&orderby=published&client=ytapi-youtube-profile", # Superinteressante (Legacy API)
    "http://www.insoonia.com/feed", # iNSôÔNiA
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCjcEWCkuafL02yvkujCwL4w", # Colecionadores de Ossos (YT)
    "https://www.youtube.com/feeds/videos.xml?channel_id=UC9li9QbQlmExYM6UxnvhuCQ", # That Creepy Reading
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCj0O6W8yDuLg3iraAXKgCrQ", # Você Sabia?
    "http://canalnostalgia.blogspot.com/feeds/posts/default", # Canal Nostalgia (Blog)
    "http://mundoestranho.abril.com.br/rss", # Mundo Estranho – Super
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCmUVDdwB1_RrQyfu80fvRrw", # Revista Galileu
    "https://www.techtudo.com.br/rss/techtudo/", # techtudo
    "http://gdata.youtube.com/feeds/base/users/iberethenorio/uploads?alt=rss&v=2&orderby=published&client=ytapi-youtube-profile", # Manual do Mundo (Legacy API)
    "http://www.assombrado.com.br/feeds/posts/default", # Assombrado
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCIQPHl1WKKTt9KkWyo_JNig", # INCRÍVEL
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCKHhA5hN2UohhFDfNXB_cvQ", # Manual do Mundo
    "https://psxbrasil.com.br/feed/", # PSX Brasil
    "http://br.ign.com/feed.xml", # IGN Brasil
    "http://gdata.youtube.com/feeds/base/users/canalmegacurioso/uploads?alt=rss&v=2&orderby=published&client=ytapi-youtube-profile", # Mega Curioso (Legacy API)
    "http://misteriosdomundo.org/feed/", # Misterios do Mundo
    "http://www.gizmodo.com.br/feed/", # Gizmodo em português
    "http://www.eltiempo.com/contenido/mundo-curioso/rss.xml", # EL TIEMPO
    "http://spacetoday.com.br/feed/", # SPACE TODAY
    "https://revistagalileu.globo.com/rss/galileu", # galileu
    "http://gdata.youtube.com/feeds/base/users/fecastanhari/uploads?alt=rss&v=2&orderby=published&client=ytapi-youtube-profile", # Canal Nostalgia (Legacy API)
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCsg11Ullu0dBrNXTrDBKOXA", # Mega Curioso
    "http://www.baratonta.com/feeds/posts/default", # Baratonta
    "http://www.bbc.co.uk/portuguese/index.xml", # BBC Brasil
    "http://super.abril.com.br/feed/", # Super
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCn9Erjy00mpnWeLnRqhsA1g", # Ciência Todo Dia
    "https://www.youtube.com/feeds/videos.xml?channel_id=UCvKP7V9o8xfB-hpk3noKAiw", # Acervo do Terror
    "http://www.hypeness.com.br/feed/", # Hypeness
    "https://www.youtube.com/playlist?list=UUn9Erjy00mpnWeLnRqhsA1g" # Ciência Todo Dia (Playlist URL)
]

# --- 2. HELPERS ---
def parse_date(date_str):
    if not date_str:
        return datetime(1970, 1, 1, tzinfo=timezone.utc)
    try:
        dt = parsedate_to_datetime(date_str)
        if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)
    except:
        try:
            clean_date = date_str.replace('Z', '+00:00')
            dt = datetime.fromisoformat(clean_date)
            if dt.tzinfo is None: dt = dt.replace(tzinfo=timezone.utc)
            return dt.astimezone(timezone.utc)
        except: return datetime(1970, 1, 1, tzinfo=timezone.utc)

def fetch_rss_items(url):
    items = []
    headers = {'User-Agent': 'Mozilla/5.0'}
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as response:
            root = ET.fromstring(response.read())
            entries = root.findall('.//item') + root.findall('.//{http://www.w3.org/2005/Atom}entry')
            for item in entries:
                title = item.findtext('title') or item.findtext('{http://www.w3.org/2005/Atom}title') or "No Title"
                # Improved Link Handling
                link = "#"
                link_node = item.find('{http://www.w3.org/2005/Atom}link')
                if link_node is not None: link = link_node.get('href')
                else: link = item.findtext('link') or "#"

                raw_date = (item.findtext('pubDate') or
                            item.findtext('{http://www.w3.org/2005/Atom}published') or
                            item.findtext('{http://www.w3.org/2005/Atom}updated'))
                items.append({"title": title, "link": link, "date": parse_date(raw_date)})
    except: pass
    return items

# --- 3. MAIN EXECUTION ---
print("--- FETCHING ARTICLES ---")
all_articles = []
for feed in RSS_FEEDS:
    all_articles.extend(fetch_rss_items(feed))

# Sort and get 100 most recent
all_articles.sort(key=lambda x: x['date'], reverse=True)
top_100_pool = all_articles[:100]

if top_100_pool:
    print(f"Ranking {len(top_100_pool)} articles with AI...")

    # Prepare the prompt for a single call
    titles_payload = "\n".join([f"ID:{i} | {a['title']}" for i, a in enumerate(top_100_pool)])
    prompt = f"Rate these articles 0-100 based on: {USER_INTERESTS}. Return ONLY JSON: {{'scores': [{{'id': 0, 'score': 85}}]}}. \nARTICLES:\n{titles_payload}"

    # Your Working URL/Model
    model_name = "gemini-2.5-flash"
    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent?key={GEMINI_API_KEY}"

    try:
        req = urllib.request.Request(url, data=json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode('utf-8'), headers={"Content-Type": "application/json"})
        with urllib.request.urlopen(req) as response:
            result = json.loads(response.read().decode('utf-8'))
            raw_text = result['candidates'][0]['content']['parts'][0]['text']
            clean_text = re.sub(r"```json|```", "", raw_text).strip()
            scores = json.loads(clean_text).get('scores', [])

            # Map scores back
            score_map = {item.get('id'): item.get('score', 0) for item in scores}
            for i, art in enumerate(top_100_pool):
                art['score'] = score_map.get(i, 0)
    except Exception as e:
        print(f"Ranking Error: {e}")

    # Sort by relevance
    top_100_pool.sort(key=lambda x: x.get('score', 0), reverse=True)

    print("\n" + "="*60)
    print("RANKED RESULTS (All 100)")
    print("="*60)
    for i, art in enumerate(top_100_pool, 1):
        print(f"{i}. [{art.get('score', 0)}/100] {art['title']}")
        print(f"   Link: {art['link']}\n")
else:
    print("No articles found.")